# Telco Customer Churn Prediction
## Exploratory notebook sandbox

This notebook is an exploratory sandbox for Machine Learning Operations (MLOps) development.

Use it to
- Inspect raw and cleaned data
- Perform focused Exploratory Data Analysis (EDA) checks to validate assumptions
- Experiment with feature settings (the `SETTINGS` dictionary below)
- Run quick training and inference checks while iterating on `src/` modules

Do not use it to
- Replace the reproducible pipeline in `src/main.py`
- Store production logic in notebook cells
- Write production artifacts to disk from the notebook

Canonical production entry point
- Run from terminal: `python -m src.main`

Why this separation matters
- The orchestrator (`src/main.py`) is the factory: deterministic inputs, deterministic outputs
- This notebook is the lab: rapid iteration, inspection, and experimentation


### A) Environment setup and imports

Learning intent
- Make imports reliable regardless of how you opened Jupyter
- Ensure relative paths behave the same way for everyone
- Keep notebook code thin by calling functions from `src/` modules


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from __future__ import annotations

import os  # For changing the working directory to the repo root
import sys  # For manipulating sys.path to ensure imports work regardless of where the notebook is started
from pathlib import Path  # For working with file paths in a platform-independent way

import pandas as pd
from IPython.display import display  # For displaying dataframes nicely in Jupyter

# Repo root alignment
# Problem this solves
# - Notebooks can start with different working directories depending on IDE and settings
# - Relative paths then break unpredictably
#
# Approach
# - Search upward from the current folder until we find a folder that contains `src/`
# - Set that as the working directory so relative paths match the orchestrator behaviour

def find_repo_root(start: Path, marker_dir: str = "src", max_hops: int = 12) -> Path:
    current = start.resolve()
    for _ in range(max_hops):
        if (current / marker_dir).exists():
            return current
        if current.parent == current:
            break
        current = current.parent
    raise RuntimeError(
        f"Could not find repo root containing '{marker_dir}/' starting from: {start}"
    )

PROJECT_ROOT = find_repo_root(Path.cwd())
os.chdir(PROJECT_ROOT)

# Ensure `import src...` works even if Jupyter started elsewhere
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("cwd:", Path.cwd())

# Imports from production modules
# Note: we intentionally do NOT import from src/main.py
from src.load_data import load_raw_data
from src.clean_data import clean_dataframe
from src.validate import validate_dataframe
from src.features import get_feature_preprocessor
from src.train import train_model
from src.evaluate import evaluate_model
from src.infer import run_inference


PROJECT_ROOT: /Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rincon@student.ie.edu/My Drive/term 2/MLops/1-mlops-kickoff-repo
cwd: /Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rincon@student.ie.edu/My Drive/term 2/MLops/1-mlops-kickoff-repo


### B) Sandbox configuration

Guideline
- Keep configuration in one place
- Use the same column names and defaults as the orchestrator where possible
- Prefer explicit lists over clever inference so errors are actionable


In [3]:
# Dataset path (same default as the orchestrator)
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

# Minimal configuration for experimentation
# - Change these lists to try different feature recipes
# - Keep the split gate early to avoid leakage

SETTINGS = {
    "target_column": "Churn",
    "problem_type": "classification",

    # Audit vault behaviour (teaching point)
    # - Keep this False while iterating
    # - Flip to True only when you are confident
    "evaluate_on_test": False,

    "split": {"test_size": 0.2, "val_size": 0.2, "random_state": 42},

    "features": {
        # Numeric columns to bin by quantiles
        "quantile_bin": ["tenure", "MonthlyCharges", "TotalCharges"],

        # Pass-through numeric features (no binning)
        "numeric_passthrough": ["SeniorCitizen"],

        # Categorical columns to one-hot encode
        "categorical_onehot": [
            "gender",
            "Partner",
            "Dependents",
            "PhoneService",
            "MultipleLines",
            "InternetService",
            "OnlineSecurity",
            "OnlineBackup",
            "DeviceProtection",
            "TechSupport",
            "StreamingTV",
            "StreamingMovies",
            "Contract",
            "PaperlessBilling",
            "PaymentMethod",
        ],

        # Optional: only pass if your get_feature_preprocessor supports it
        "n_bins": 5,
    },

    # Validation contract (used depending on validate_dataframe signature)
    "schema": {
        "gender": {"type": "categorical", "accept_nan": False},
        "SeniorCitizen": {"type": "numeric", "accept_nan": False},
        "Partner": {"type": "categorical", "accept_nan": False},
        "Dependents": {"type": "categorical", "accept_nan": False},
        "tenure": {"type": "numeric", "accept_nan": False},
        "PhoneService": {"type": "categorical", "accept_nan": False},
        "MultipleLines": {"type": "categorical", "accept_nan": False},
        "InternetService": {"type": "categorical", "accept_nan": False},
        "OnlineSecurity": {"type": "categorical", "accept_nan": False},
        "OnlineBackup": {"type": "categorical", "accept_nan": False},
        "DeviceProtection": {"type": "categorical", "accept_nan": False},
        "TechSupport": {"type": "categorical", "accept_nan": False},
        "StreamingTV": {"type": "categorical", "accept_nan": False},
        "StreamingMovies": {"type": "categorical", "accept_nan": False},
        "Contract": {"type": "categorical", "accept_nan": False},
        "PaperlessBilling": {"type": "categorical", "accept_nan": False},
        "PaymentMethod": {"type": "categorical", "accept_nan": False},
        "MonthlyCharges": {"type": "numeric", "accept_nan": False},
        "TotalCharges": {"type": "numeric", "accept_nan": False},
    },
    "target_config": {
        "column": "Churn",
        "type": "classification",
        "allowed_classes": [0, 1],
    },
    "validation": {
        "check_missing_values": True,
        "numeric_non_negative_cols": ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"],
    },

    # Optional: if your src.train.train_model supports hyperparameter search
    # "param_grid": {
    #     "model__max_depth": [3, 4, 6],
    #     "model__n_estimators": [200, 500],
    # },
}


def three_way_split(
    X: pd.DataFrame,
    y: pd.Series,
    *,
    test_size: float,
    val_size: float,
    random_state: int,
    stratify: bool,
):
    """
    Sandbox copy aligned with the orchestrator intent
    - Students benefit from seeing the leakage gate explicitly
    - We avoid importing helpers from src/main.py to prevent side effects

    Behaviour
    - Try stratified splits for classification
    - If stratification fails, fall back to random splits with a clear message
    """
    from sklearn.model_selection import train_test_split

    if test_size <= 0 or val_size <= 0 or (test_size + val_size) >= 1.0:
        raise ValueError("Split sizes must satisfy 0 < test_size, 0 < val_size, and test_size + val_size < 1")

    stratify_y = y if stratify else None

    try:
        X_temp, X_test, y_temp, y_test = train_test_split(
            X,
            y,
            test_size=test_size,
            random_state=random_state,
            stratify=stratify_y,
        )

        relative_val_size = val_size / (1.0 - test_size)
        stratify_temp = y_temp if stratify else None

        X_train, X_val, y_train, y_val = train_test_split(
            X_temp,
            y_temp,
            test_size=relative_val_size,
            random_state=random_state,
            stratify=stratify_temp,
        )

        return X_train, X_val, X_test, y_train, y_val, y_test

    except ValueError as e:
        print(f"[notebook] Stratified split failed: {e}")
        print("[notebook] Falling back to random split")

        X_temp, X_test, y_temp, y_test = train_test_split(
            X,
            y,
            test_size=test_size,
            random_state=random_state,
        )

        relative_val_size = val_size / (1.0 - test_size)

        X_train, X_val, y_train, y_val = train_test_split(
            X_temp,
            y_temp,
            test_size=relative_val_size,
            random_state=random_state,
        )

        return X_train, X_val, X_test, y_train, y_val, y_test


print("RAW_DATA_PATH:", RAW_DATA_PATH)
print("TARGET:", SETTINGS["target_column"])
print("PROBLEM_TYPE:", SETTINGS["problem_type"])


RAW_DATA_PATH: /Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rincon@student.ie.edu/My Drive/term 2/MLops/1-mlops-kickoff-repo/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv
TARGET: Churn
PROBLEM_TYPE: classification


### 1) Load raw data (`src.load_data`)

Educational note
- We load raw data exactly once
- Raw data should be treated as immutable input


In [4]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Raw data not found at {RAW_DATA_PATH}\n"
        "Check that you opened the correct repository folder and that data/raw exists"
    )

df_raw = load_raw_data(RAW_DATA_PATH)
print("df_raw.shape:", df_raw.shape)
display(df_raw.head())


[load_data.load_raw_data] Loading raw data from: /Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rincon@student.ie.edu/My Drive/term 2/MLops/1-mlops-kickoff-repo/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv
df_raw.shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### 2) Focused EDA checks

Goal
- Validate assumptions before investing in feature engineering and training

Guideline
- Prefer small, targeted checks over long notebooks
- If an assumption is wrong, fix the pipeline, not the notebook


In [5]:
print("Missing values (top 10 columns):")
display(df_raw.isna().sum().sort_values(ascending=False).head(10))

# Target distribution (raw)
target_col = SETTINGS["target_column"]
if target_col in df_raw.columns:
    print("\nTarget distribution (raw):")
    display(df_raw[target_col].value_counts(dropna=False))
    print("\nTarget prevalence (raw):")
    display(df_raw[target_col].value_counts(normalize=True, dropna=False))
else:
    print(f"Target column '{target_col}' not found in raw data")

# Telco teaching point: TotalCharges often comes as a string with blanks
if "TotalCharges" in df_raw.columns:
    blank_total = df_raw["TotalCharges"].astype(str).str.strip().eq("").sum()
    print(f"\nTotalCharges blank/whitespace rows: {blank_total}")

    total_num = pd.to_numeric(df_raw["TotalCharges"], errors="coerce")
    print("\nTotalCharges (as numeric, coercing blanks -> NaN):")
    display(total_num.describe())

# Quick look at key numeric columns
for col in ["tenure", "MonthlyCharges"]:
    if col in df_raw.columns:
        print(f"\nSummary for '{col}':")
        display(df_raw[col].describe())


Missing values (top 10 columns):


customerID          0
DeviceProtection    0
TotalCharges        0
MonthlyCharges      0
PaymentMethod       0
PaperlessBilling    0
Contract            0
StreamingMovies     0
StreamingTV         0
TechSupport         0
dtype: int64


Target distribution (raw):


Churn
No     5174
Yes    1869
Name: count, dtype: int64


Target prevalence (raw):


...

count    7032.000000
mean     2283.300441
std      2266.771362
min        18.800000
25%       401.450000
50%      1397.475000
75%      3794.737500
max      8684.800000
Name: TotalCharges, dtype: float64


Summary for 'tenure':


count    7043.000000
mean       32.371149
std        24.559481
min         0.000000
25%         9.000000
50%        29.000000
75%        55.000000
max        72.000000
Name: tenure, dtype: float64


Summary for 'MonthlyCharges':


count    7043.000000
mean       64.761692
std        30.090047
min        18.250000
25%        35.500000
50%        70.350000
75%        89.850000
max       118.750000
Name: MonthlyCharges, dtype: float64

### 3) Clean data (`src.clean_data`)

Educational note
- Cleaning should be deterministic
- Cleaning should not learn from the data in a way that leaks information across splits


In [6]:
df_clean = clean_dataframe(df_raw, target_column=SETTINGS["target_column"])
print("df_clean.shape:", df_clean.shape)
display(df_clean.head())


[clean_data.clean_dataframe] Cleaning dataframe (baseline: identity copy)
[clean_data.clean_dataframe] Dropping high-cardinality/ID columns: ['customerID']
df_clean.shape: (7043, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0.0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0.0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1.0
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0.0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1.0


### 4) Didactic check: what changed after cleaning

Goal
- Make transformations visible
- Help you debug unexpected column name changes or dropped columns


In [7]:
raw_cols = list(df_raw.columns)
clean_cols = list(df_clean.columns)

removed_cols = sorted(set(raw_cols) - set(clean_cols))
added_cols = sorted(set(clean_cols) - set(raw_cols))

print("Columns removed:")
display(removed_cols)

print("Columns added:")
display(added_cols)

# Type checks for teaching/debugging
if "TotalCharges" in df_clean.columns:
    print("\nTotalCharges dtype after cleaning:", df_clean["TotalCharges"].dtype)

if SETTINGS["target_column"] in df_clean.columns:
    print("\nTarget unique values after cleaning:")
    display(df_clean[SETTINGS["target_column"]].value_counts(dropna=False))


Columns removed:


['customerID']

Columns added:


[]


TotalCharges dtype after cleaning: float64

Target unique values after cleaning:


Churn
0.0    5174
1.0    1869
Name: count, dtype: int64

### 5) Validate data (security gate)

Educational note
- Validation is a fail-fast gate
- It prevents wasting time on training with broken assumptions

Guideline
- If validation fails, fix the upstream module or the data contract
- Do not patch around failures inside the notebook


In [8]:
import inspect

# Build a "required columns" list for validation functions that use that contract
required_columns = (
    [SETTINGS["target_column"]]
    + SETTINGS["features"]["quantile_bin"]
    + SETTINGS["features"]["categorical_onehot"]
    + SETTINGS["features"]["numeric_passthrough"]
)
# Deduplicate while preserving order
required_columns = list(dict.fromkeys(required_columns))

# Support both validate_dataframe contracts (schema-based and required_columns-based)
val_sig = inspect.signature(validate_dataframe)

if "schema" in val_sig.parameters:
    validate_dataframe(
        df=df_clean,
        schema=SETTINGS["schema"],
        target_config=SETTINGS["target_config"],
    )
else:
    validate_dataframe(
        df=df_clean,
        required_columns=required_columns,
        check_missing_values=SETTINGS["validation"].get("check_missing_values", True),
        target_column=SETTINGS["target_column"],
        target_allowed_values=[0, 1] if SETTINGS["problem_type"] == "classification" else None,
        numeric_non_negative_cols=SETTINGS["validation"].get("numeric_non_negative_cols", None),
    )

print("[notebook] Validation passed")


[validate] Running data quality checks...
[validate] PASSED  — Shape: 7,043 rows × 20 columns
[validate] PASSED  — No fully-NaN rows found
[validate] PASSED  — Schema OK for 19 column(s) (NaN policy enforced per column)
[validate] PASSED  — Target column 'Churn' present
[validate] PASSED  — Target classes {np.float64(0.0), np.float64(1.0)} ⊆ allowed {0, 1}
[validate] PASSED  — No negative values across 5 numeric column(s)
[validate] All checks passed ✓
[notebook] Validation passed


### 6) Build the feature recipe (`src.features`)

Educational note
- This step builds a preprocessing blueprint
- It must not fit on the full dataset in the notebook
- The recipe learns only when fitted on the training split inside the training pipeline


In [9]:
import inspect

feat_sig = inspect.signature(get_feature_preprocessor)

# Build kwargs defensively (only pass what the function accepts)
feat_kwargs = {
    "quantile_bin_cols": SETTINGS["features"]["quantile_bin"],
    "categorical_onehot_cols": SETTINGS["features"]["categorical_onehot"],
    "numeric_passthrough_cols": SETTINGS["features"]["numeric_passthrough"],
}

if "n_bins" in feat_sig.parameters and "n_bins" in SETTINGS["features"]:
    feat_kwargs["n_bins"] = SETTINGS["features"]["n_bins"]

# Keep only accepted kwargs
feat_kwargs = {k: v for k, v in feat_kwargs.items() if k in feat_sig.parameters}

preprocessor = get_feature_preprocessor(**feat_kwargs)

print("[notebook] Feature recipe built (not fitted yet)")
preprocessor


[notebook] Feature recipe built (not fitted yet)


,transformers,"[('numeric', ...), ('tenure_risk_bucket', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,func,<function saf...t 0x12f24d480>
,inverse_func,None
,validate,False


### 7) Leakage gate: three-way split (train, validation, test)

Educational note
- Train split is the only split allowed to learn preprocessing parameters and model weights
- Validation split is for iteration and model selection
- Test split is a final audit vault


In [10]:
X = df_clean.drop(columns=[SETTINGS["target_column"]])
y = df_clean[SETTINGS["target_column"]]

if SETTINGS["problem_type"] == "classification" and y.nunique() < 2:
    raise ValueError(
        f"Target '{SETTINGS['target_column']}' has fewer than 2 classes after cleaning\n"
        "Classification training and stratified splitting cannot proceed"
    )

X_train, X_val, X_test, y_train, y_val, y_test = three_way_split(
    X,
    y,
    test_size=SETTINGS["split"]["test_size"],
    val_size=SETTINGS["split"]["val_size"],
    random_state=SETTINGS["split"]["random_state"],
    stratify=(SETTINGS["problem_type"] == "classification"),
)

print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)

print("\nClass balance (train):")
display(y_train.value_counts(normalize=True))
print("\nClass balance (val):")
display(y_val.value_counts(normalize=True))
print("\nClass balance (test):")
display(y_test.value_counts(normalize=True))


Train: (4225, 19) Validation: (1409, 19) Test: (1409, 19)

Class balance (train):


Churn
0.0    0.734675
1.0    0.265325
Name: proportion, dtype: float64


Class balance (val):


Churn
0.0    0.734564
1.0    0.265436
Name: proportion, dtype: float64


Class balance (test):


Churn
0.0    0.734564
1.0    0.265436
Name: proportion, dtype: float64

### 8) Train (`src.train`)

Educational note
- This is the only step where `.fit()` happens
- The preprocessor and estimator are fitted only on training data


In [11]:
import inspect

train_sig = inspect.signature(train_model)

train_kwargs = {
    "X_train": X_train,
    "y_train": y_train,
    "preprocessor": preprocessor,
    "problem_type": SETTINGS["problem_type"],
}

# Optional hyperparameter search support
if "param_grid" in train_sig.parameters and SETTINGS.get("param_grid") is not None:
    train_kwargs["param_grid"] = SETTINGS["param_grid"]

# Keep only accepted kwargs
train_kwargs = {k: v for k, v in train_kwargs.items() if k in train_sig.parameters}

model_pipeline = train_model(**train_kwargs)

print("[notebook] Training complete")
model_pipeline


[train] Starting model training | problem_type='classification'
[train] WARNING: 'model__max_depth' not in param_grid — using default values: [3, 6]
[train] WARNING: 'model__learning_rate' not in param_grid — using default values: [0.01, 0.15]
[train] WARNING: 'model__n_estimators' not in param_grid — using default values: [100, 300]
[train] WARNING: 'model__subsample' not in param_grid — using default values: [0.7, 0.9]
[train] WARNING: 'model__colsample_bytree' not in param_grid — using default values: [0.7, 0.9]
[train] WARNING: 'model__gamma' not in param_grid — using default values: [0, 0.1]
Fitting 5 folds for each of 64 candidates, totalling 320 fits
[train] Best Parameters: {'model__colsample_bytree': 0.9, 'model__gamma': 0, 'model__learning_rate': 0.15, 'model__max_depth': 3, 'model__n_estimators': 100, 'model__subsample': 0.9}
[train] Best CV F1: 0.5834
[notebook] Training complete


,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('tenure_risk_bucket', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### 9) Evaluate (`src.evaluate`)

**Educational Note: The Audit Vault**
- **Validation:** Used to iteratively guide our model and feature engineering decisions.
- **Test:** Our blind audit vault. We use a strict boolean flag (`evaluate_on_test` in our `SETTINGS`) to ensure we only peek at this vault when we are absolutely ready to finalize the model.

Metric note
- `evaluate_model` should select appropriate metrics for the `problem_type`.


In [12]:
import inspect

ev_sig = inspect.signature(evaluate_model)

def _eval(model, X_eval, y_eval):
    # Support both contracts used across different cohorts
    if "X_eval" in ev_sig.parameters:
        return evaluate_model(
            model=model,
            X_eval=X_eval,
            y_eval=y_eval,
            problem_type=SETTINGS["problem_type"],
        )
    else:
        return evaluate_model(
            model=model,
            X_test=X_eval,
            y_test=y_eval,
            problem_type=SETTINGS["problem_type"],
        )

val_metrics = _eval(model_pipeline, X_val, y_val)
print(f"[notebook] Validation metrics: {val_metrics}")

if SETTINGS.get("evaluate_on_test", False):
    test_metrics = _eval(model_pipeline, X_test, y_test)
    print(f"[notebook] Test metrics (audit vault): {test_metrics}")
else:
    print("[notebook] Test metrics: Skipped to protect the audit vault. Set 'evaluate_on_test': True to run.")


[notebook] Validation metrics: 0.5645645645645646
[notebook] Test metrics: Skipped to protect the audit vault. Set 'evaluate_on_test': True to run.


### 10) Inference demo (`src.infer`)

Educational note
- Inference simulates what happens after training
- We deliberately use rows from the test split to simulate unseen cases


In [15]:
import inspect

inf_sig = inspect.signature(run_inference)

sample_n = min(10, len(X_test))
X_infer_sample = X_test.sample(n=sample_n, random_state=SETTINGS["split"]["random_state"])

inf_kwargs = {
    "model": model_pipeline,
    "X_infer": X_infer_sample,
}

# Optional include_proba support (classification demos often include probabilities)
if "include_proba" in inf_sig.parameters:
    inf_kwargs["include_proba"] = (SETTINGS["problem_type"] == "classification")

inf_kwargs = {k: v for k, v in inf_kwargs.items() if k in inf_sig.parameters}

df_predictions = run_inference(model_pipeline,X_infer_sample)

print("[notebook] Inference results")
display(df_predictions.head(10))


[notebook] Inference results


,prediction,proba
5902,0,0.162183
5415,0,0.299866
2874,0,0.005259
2686,1,0.565661
700,0,0.025298
4114,0,0.004834
2967,0,0.074267
3158,0,0.161969
4849,0,0.326300
2246,1,0.878497


### 11) Inspect production artifacts produced by the orchestrator

This notebook does not write artifacts to disk.

Use this cell only after you run the orchestrator from terminal
- `python -m src.main`

Goal
- Prove that the factory output exists on disk
- Inspect outputs without modifying them


In [ ]:
from src.utils import load_model

CLEAN_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "clean.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "model.joblib"
PREDICTIONS_PATH = PROJECT_ROOT / "reports" / "predictions.csv"

try:
    clean_from_disk = pd.read_csv(CLEAN_DATA_PATH)
    preds_from_disk = pd.read_csv(PREDICTIONS_PATH)
    model_from_disk = load_model(MODEL_PATH)

    print("clean.csv shape:", clean_from_disk.shape)
    print("predictions.csv shape:", preds_from_disk.shape)
    print("loaded model type:", type(model_from_disk))

    display(preds_from_disk.head(10))

except Exception as e:
    print("Artifacts not found yet or could not be loaded")
    print("Run from terminal: python -m src.main")
    print("Error:", e)
